# Reproducibility notebook
## Structure-Preserving Fourth-Order Lie-Group Integration and Noise-Metric-Aware Optimal Control for Random Schrödinger Equations
S. Vijayaram and P. Balasubramaniam — revised manuscript for *SciPost Physics Core*

This single notebook reproduces **every numerical result, figure and table** of the revised manuscript. Run the blocks from top to bottom (*Kernel → Restart & Run All*).

**Requirements:** Python ≥ 3.9 with `numpy`, `scipy`, `matplotlib`. Block 16 (two-qubit CZ gate) also needs `jax` (`pip install "jax[cpu]"`). If JAX is missing, Block 16 is skipped and everything else still runs.

**Outputs:** all figures (PDF + PNG, with the file names used in the LaTeX source), tables (CSV) and result files (JSON) are written to the folder `rse_outputs/` next to this notebook. Optimizations are checkpointed there, so a second run is fast.

**Runtime:** about 20–40 minutes on a laptop with `FAST = False` (Block 1). `FAST = True` gives a smoke test in a few minutes; its numbers will **not** match the paper.

| Block | Content | Manuscript |
|---|---|---|
| 1–5 | Settings, model, CFM4 propagator, self-tests, noise model and optimizer | Secs. 3–5 |
| 6 | Convergence: smooth Hamiltonian and rough Matérn path | Fig. 2, Remark 4.1 |
| 7 | Training of the four strategies (noise-blind, isotropic, anisotropic, ensemble) | Sec. 6.4 |
| 8 | Monte Carlo validation, histograms, controls, baseline comparison | Table 1, Figs. 4, 5, 6 |
| 9 | Unitarity: CFM4 vs RK4 | Fig. 1 |
| 10 | Pareto frontier and robustness-bound test | Figs. 3, 8 |
| 11 | Noise-amplitude and correlation-length sweeps | Figs. 9, 10 |
| 12 | Anisotropy sweep | Figs. 12, 13 |
| 13–14 | Discrete adjoint gradients: verification, cost, re-optimization | Sec. 6.5, Fig. 7 |
| 15 | Additive dephasing scenario | Sec. 6.10, Fig. 14 |
| 16 | Two-qubit controlled-Z gate (JAX) | Sec. 6.8, Fig. 11 |
| 17 | Comparison of the reproduced numbers with the values reported in the manuscript | — |

All random seeds are fixed. Validation noise uses seeds disjoint from all training noise.

## Block 1 — Imports and global settings

In [ ]:
import os, json, time
import numpy as np
from scipy.linalg import expm
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline

FAST = False          # True: quick smoke test (numbers will NOT match the paper)
RUN_CZ = True         # Block 16 needs JAX
OUT = os.path.abspath('rse_outputs'); os.makedirs(OUT, exist_ok=True)

plt.rcParams.update({'font.family':'serif','font.size':12,'axes.labelsize':14,
 'legend.fontsize':11,'xtick.labelsize':11,'ytick.labelsize':11,'figure.dpi':110,
 'savefig.dpi':300,'savefig.bbox':'tight','mathtext.fontset':'cm',
 'axes.grid':True,'grid.alpha':0.5,'grid.linestyle':'--'})

BOX_LABELS = ['noise-blind\n$\\lambda=0$', 'ensemble\nrobust GRAPE', 'isotropic\n$l_y=1$', 'anisotropic\n$l_y=4$']
def savefig(name):
    plt.savefig(f'{OUT}/{name}.pdf'); plt.savefig(f'{OUT}/{name}.png')
    plt.show(); print('saved', name)

ITERS = 45 if FAST else 450        # Adam iterations, deterministic objectives
ITERS_D = 30 if FAST else 300      # Adam iterations, 30-sample ensemble objective
print('numpy', np.__version__, '| output folder:', OUT, '| FAST =', FAST)

## Block 2 — Single-qubit model, CFM4 coefficients and Matérn noise sampler

In [ ]:
T = 5.0; N = 200; h = T/N; omega0 = 1.0
tgrid = np.linspace(0, T, N+1)
c1, c2 = 0.5-np.sqrt(3)/6.0, 0.5+np.sqrt(3)/6.0          # Gauss-Legendre nodes
a1, a2 = 0.25+np.sqrt(3)/6.0, 0.25-np.sqrt(3)/6.0        # CFM4 coefficients (a1+a2 = 1/2)
tn1 = tgrid[:-1]+c1*h; tn2 = tgrid[:-1]+c2*h
theta = np.pi/2                                          # target V = R_x(pi/2)
V = np.array([[np.cos(theta/2), -1j*np.sin(theta/2)],
              [-1j*np.sin(theta/2), np.cos(theta/2)]])
I2 = np.eye(2, dtype=complex)
sx = np.array([[0,1],[1,0]], complex); sy = np.array([[0,-1j],[1j,0]], complex)
sz = np.array([[1,0],[0,-1]], complex)

def matern_cov(ts, sigma, ell):
    r_ = np.abs(ts[:,None]-ts[None,:]); f = np.sqrt(3.0)*r_/ell
    return sigma**2*(1.0+f)*np.exp(-f) + 1e-10*np.eye(len(ts))
def sample_matern(rng, ts, sigma, ell, S):
    L = np.linalg.cholesky(matern_cov(ts, sigma, ell))
    return (L @ rng.standard_normal((len(ts), S))).T

ts_nodes = np.sort(np.concatenate([tn1, tn2]))           # noise is sampled at the quadrature nodes
idx1 = np.searchsorted(ts_nodes, tn1); idx2 = np.searchsorted(ts_nodes, tn2)
print('a1 + a2 =', a1+a2)

## Block 3 — Batched CFM4 propagator and cost functions
Closed-form SU(2) exponentials; vectorized over control batches `B` and noise samples `S`.

In [ ]:
def prop_cfm4(ux, uy, nz, ndx, ndy, return_traj=False):
    # nz : additive sigma_z noise; ndx, ndy : relative amplitude noise u -> u(1+delta).
    # Each is None or a pair (values at node 1, values at node 2), each of shape (S, N).
    ux = np.atleast_2d(ux); uy = np.atleast_2d(uy); B = ux.shape[0]
    pair = lambda p: None if p is None else (np.atleast_2d(p[0]), np.atleast_2d(p[1]))
    nz, ndx, ndy = pair(nz), pair(ndx), pair(ndy)
    S = 1
    for p in (nz, ndx, ndy):
        if p is not None: S = max(S, p[0].shape[0])
    z1 = 0.5*omega0 + (nz[0] if nz is not None else 0.0)
    z2 = 0.5*omega0 + (nz[1] if nz is not None else 0.0)
    z1 = np.broadcast_to(np.atleast_2d(z1), (S, N))[None]
    z2 = np.broadcast_to(np.atleast_2d(z2), (S, N))[None]
    ux_ = ux[:, None, :]; uy_ = uy[:, None, :]
    fx1 = 1.0 + (ndx[0][None] if ndx is not None else 0.0); fx2 = 1.0 + (ndx[1][None] if ndx is not None else 0.0)
    fy1 = 1.0 + (ndy[0][None] if ndy is not None else 0.0); fy2 = 1.0 + (ndy[1][None] if ndy is not None else 0.0)
    o1 = ux_*fx1 - 1j*uy_*fy1; o2 = ux_*fx2 - 1j*uy_*fy2
    Es = []
    for (wa, wb) in ((a1, a2), (a2, a1)):                 # Omega_1, then Omega_2
        w = -1j*h*(wa*z1 + wb*z2); o = -1j*h*(wa*o1 + wb*o2)
        v = np.sqrt(np.abs(w)**2 + np.abs(o)**2); cv = np.cos(v)
        vs = np.where(v > 1e-14, v, 1.0); sv = np.where(v > 1e-14, np.sin(vs)/vs, 1.0)
        Es.append((cv+sv*w, sv*o, -sv*np.conj(o), cv-sv*w))
    U00 = np.ones((B,S), complex); U01 = np.zeros((B,S), complex)
    U10 = np.zeros((B,S), complex); U11 = np.ones((B,S), complex); traj = []
    for n in range(N):
        for (E00, E01, E10, E11) in Es:
            a_ = np.broadcast_to(E00[...,n], (B,S)); b_ = np.broadcast_to(E01[...,n], (B,S))
            c_ = np.broadcast_to(E10[...,n], (B,S)); d_ = np.broadcast_to(E11[...,n], (B,S))
            U00, U01, U10, U11 = a_*U00+b_*U10, a_*U01+b_*U11, c_*U00+d_*U10, c_*U01+d_*U11
        if return_traj: traj.append((U00.copy(), U01.copy(), U10.copy(), U11.copy()))
    return traj if return_traj else (U00, U01, U10, U11)

def noise_pairs(raw, scale, clip=3.0):
    x = np.clip(raw, -clip, clip)*scale                   # truncation at 3 sigma (essential boundedness)
    return (x[:, idx1], x[:, idx2])

def infid(U00, U01, U10, U11):                            # 1 - |tr(V^dag U)|/2
    tr = np.conj(V[0,0])*U00+np.conj(V[1,0])*U01+np.conj(V[0,1])*U10+np.conj(V[1,1])*U11
    return 1.0 - np.abs(tr)/2.0

def dF(U00, U01, U10, U11, W00, W01, W10, W11):           # ||log(W^dag U)||_F on SU(2)
    tr = np.conj(W00)*U00+np.conj(W01)*U01+np.conj(W10)*U10+np.conj(W11)*U11
    return np.sqrt(2.0)*np.arccos(np.clip(np.real(tr)/2.0, -1.0, 1.0))

def Jnoise(ux, uy, lx, ly): return h*np.sum(np.sqrt(lx*ux**2 + ly*uy**2), axis=-1)

## Block 4 — Self-tests
The propagator must be unitary and agree with an independent dense-exponential implementation.

In [ ]:
_rng = np.random.default_rng(0)
_ux, _uy = _rng.uniform(-2, 2, N), _rng.uniform(-2, 2, N)
U = prop_cfm4(_ux, _uy, None, None, None); Um = np.array([[U[0][0,0], U[1][0,0]], [U[2][0,0], U[3][0,0]]])
Uref = I2.copy()
for n in range(N):
    H1 = 0.5*omega0*sz + _ux[n]*sx + _uy[n]*sy           # piecewise-constant controls: H(t1) = H(t2)
    Uref = expm(-1j*h*(a2*H1+a1*H1)) @ expm(-1j*h*(a1*H1+a2*H1)) @ Uref
print('unitarity defect   :', np.linalg.norm(Um.conj().T@Um - I2))
print('vs dense expm      :', np.linalg.norm(Um - Uref))
assert np.linalg.norm(Um.conj().T@Um - I2) < 1e-12 and np.linalg.norm(Um - Uref) < 1e-12
print('self-tests passed')

## Block 5 — Noise model, result store and checkpointed optimizer
`SCEN = 'M'`: control-coupled amplitude noise (main scenario). `SCEN = 'Z'`: additive dephasing (Block 15). Gradients in Blocks 7–12 are batched forward finite differences, as in the manuscript; Block 14 repeats the baseline with adjoint gradients.

In [ ]:
SIG_X, SIG_Y, SIG_Z, ELL = 0.10, 0.20, 0.05, 0.4

def set_scenario(scen):
    global SCEN, RES, Lstd, tr_noise
    SCEN = scen
    RES = f'{OUT}/results.json' if scen == 'M' else f'{OUT}/results_dephasing.json'
    Lstd = np.linalg.cholesky(matern_cov(ts_nodes, 1.0, ELL))
    tr_noise = make_noise(np.random.default_rng(12345), 30)     # 30 training realizations

def draw_std(rng, S): return (Lstd @ rng.standard_normal((len(ts_nodes), S))).T
def make_noise(rng, S, sx_=1.0, sell=None):
    if sell is not None:
        Ls = np.linalg.cholesky(matern_cov(ts_nodes, 1.0, sell))
        dr = lambda: (Ls @ rng.standard_normal((len(ts_nodes), S))).T
    else:
        dr = lambda: draw_std(rng, S)
    if SCEN == 'M':
        return (None, noise_pairs(dr(), SIG_X*sx_), noise_pairs(dr(), SIG_Y*sx_))
    return (noise_pairs(dr(), SIG_Z*sx_), None, None)

def load_res():
    try: return json.load(open(RES))
    except Exception: return {}
def save_res(r): json.dump(r, open(RES, 'w'), indent=1)

def opt_ckpt(name, lam, lx, ly, ens, iters=450, lr0=0.05):
    # Projected Adam with batched forward finite differences; checkpointed in OUT.
    f = f'{OUT}/ckpt_{SCEN}_{name}_{iters}.npz'
    nz, ndx, ndy = tr_noise if ens else (None, None, None)
    def J_batch(ub):
        Ux, Uy = ub[:, :N], ub[:, N:]
        return infid(*prop_cfm4(Ux, Uy, nz, ndx, ndy)).mean(axis=1) + lam*Jnoise(Ux, Uy, lx, ly)
    if os.path.exists(f):
        u = np.load(f)['u']
    else:
        t0 = time.perf_counter()
        u = np.concatenate([0.5*np.sin(2*np.pi*tgrid[:-1]/T), 0.5*np.cos(2*np.pi*tgrid[:-1]/T)])
        m = np.zeros(2*N); vA = np.zeros(2*N); eps = 3e-7; b1, b2 = 0.9, 0.999
        for it in range(1, iters+1):
            P = np.vstack([u[None,:] + eps*np.eye(2*N), u[None,:]])
            Jv = J_batch(P); g = (Jv[:2*N] - Jv[-1])/eps
            lr = lr0*(0.3 if it > iters*0.6 else 1.0)*(0.1 if it > iters*0.85 else 1.0)
            m = b1*m + (1-b1)*g; vA = b2*vA + (1-b2)*g*g
            u = np.clip(u - lr*(m/(1-b1**it))/(np.sqrt(vA/(1-b2**it)) + 1e-12), -2.0, 2.0)
        np.savez(f, u=u)
        print(f'  [{SCEN}/{name}] trained in {time.perf_counter()-t0:.1f} s')
    ux, uy = u[:N], u[N:]
    ci = float(infid(*prop_cfm4(ux, uy, None, None, None))[0,0])
    return ux, uy, ci

VAR = dict(A=(0.0,1,1,False), B=(0.005,1,1,False), C=(0.005,1,4,False), D=(0.0,1,1,True))
def get(name):
    lam, lx, ly, ens = VAR[name]
    return opt_ckpt(name, lam, lx, ly, ens, iters=ITERS_D if ens else ITERS)

def val_noise(): return make_noise(np.random.default_rng(777), 1000)   # disjoint validation seed
def evaluate(ux, uy, nzz): return infid(*prop_cfm4(ux, uy, *nzz))[0]
def stats(x): return dict(mean=float(x.mean()), std=float(x.std()), wc=float(x.max()))

set_scenario('M'); r = load_res()
print('scenario', SCEN, '| results file', RES)

## Block 6 — Convergence of CFM4 (Fig. 2, Remark 4.1)
Smooth Hamiltonian H(t) = sin t σx + cos t σz, and the same Hamiltonian with a rough Matérn ν = 3/2 path added; reference with N = 10⁵ steps. Takes about 1–2 minutes.

In [ ]:
def run_fig2():
    def expmH(O):
        v = np.sqrt(abs(O[0,0])**2 + abs(O[0,1])**2)
        return np.cos(v)*I2 + (np.sin(v)/v)*O if v > 1e-14 else I2
    Tc = 2.0
    def Hc(t, eta=None):
        H = np.sin(t)*sx + np.cos(t)*sz
        return H + eta(t)*sz if eta is not None else H
    def prop(Tt, Nn, eta=None):
        hh = Tt/Nn; U = I2.copy()
        for tn in np.linspace(0, Tt, Nn, endpoint=False):
            A1 = -1j*Hc(tn+c1*hh, eta); A2 = -1j*Hc(tn+c2*hh, eta)
            U = expmH(hh*(a2*A1+a1*A2)) @ expmH(hh*(a1*A1+a2*A2)) @ U
        return U
    rng = np.random.default_rng(999)
    tf = np.linspace(0, Tc, 3201)
    path = np.linalg.cholesky(matern_cov(tf, 0.05, 0.4)) @ rng.standard_normal(len(tf))
    eta = lambda t: np.interp(t, tf, path)
    Nref = 20000 if FAST else 100000
    Us, Ur = prop(Tc, Nref), prop(Tc, Nref, eta)
    Ns = [10, 16, 24, 32, 48, 64, 96, 128]; hs = [Tc/n for n in Ns]
    es = [np.linalg.norm(prop(Tc, n) - Us) for n in Ns]
    er = [np.linalg.norm(prop(Tc, n, eta) - Ur) for n in Ns]
    def fit(hv, ev):
        vi = [i for i, e in enumerate(ev) if e > 5e-10] or list(range(len(ev)))
        return np.polyfit(np.log(np.array(hv)[vi]), np.log(np.array(ev)[vi]), 1)[0]
    ss, sr = fit(hs, es), fit(hs, er)
    plt.figure(figsize=(8,5))
    plt.loglog(hs, es, 'o-', color='#1f77b4', lw=2.5, ms=8, label=f'smooth Hamiltonian (observed slope $\\approx {ss:.2f}$)')
    plt.loglog(hs, er, 's-', color='#d62728', lw=2.5, ms=8, label=f'Matérn $\\nu=3/2$ sample path (observed slope $\\approx {sr:.2f}$)')
    plt.loglog(hs, (np.array(hs)/hs[0])**4*es[0], 'k--', lw=2, label=r'$\mathcal{O}(h^4)$ reference')
    plt.loglog(hs, (np.array(hs)/hs[0])**2*er[0], 'k:', lw=2, label=r'$\mathcal{O}(h^2)$ reference')
    plt.xlabel(r'Time Step $h$'); plt.ylabel(r'Global Error $||U(T) - U_N||_F$')
    plt.title('Convergence Analysis of CFM4 Integrator'); plt.legend(loc='lower right')
    savefig('figure2_convergence')
    json.dump({'h': hs, 'err_smooth': es, 'err_rough': er, 'slope_smooth': ss, 'slope_rough': sr},
              open(f'{OUT}/fig2_data.json', 'w'), indent=1)
    print(f'observed orders: smooth {ss:.2f}, rough Matérn {sr:.2f}')
    return ss, sr
fig2_slopes = run_fig2()

## Block 7 — Training of the four control strategies (Sec. 6.4)
A: noise-blind (λ = 0). B: isotropic penalty (λ = 5×10⁻³, l_y = 1). C: anisotropic noise-metric penalty (λ = 5×10⁻³, l_y = 4). D: ensemble robust GRAPE (30 training realizations). Strategy D is the slowest block (several minutes).

In [ ]:
ctrl = {}
for nm in ['A', 'B', 'C', 'D']:
    ctrl[nm] = get(nm)
    print(f'{nm}: noiseless infidelity {ctrl[nm][2]:.3e}')

## Block 8 — Monte Carlo validation: Table 1 and Figs. 4, 5, 6

In [ ]:
vn = val_noise(); out = {}; infs = {}
for nm in ['A', 'B', 'C', 'D']:
    ux, uy, ci = ctrl[nm]; iv = evaluate(ux, uy, vn); infs[nm] = iv
    out[nm] = dict(**stats(iv), clean=ci, Jnoise_aniso=float(Jnoise(ux, uy, 1, 4)))
r['baseline'] = out; save_res(r)

labs = {'A': 'noise-blind (λ=0)', 'D': 'ensemble robust GRAPE', 'B': 'isotropic penalty (l_y=1)', 'C': 'anisotropic penalty (l_y=4)'}
print(f"{'strategy':30s} {'mean':>10s} {'std':>10s} {'worst':>10s} {'clean':>10s} {'J_noise':>8s}")
for nm in ['A', 'D', 'B', 'C']:
    o = out[nm]; print(f"{labs[nm]:30s} {o['mean']:10.3e} {o['std']:10.3e} {o['wc']:10.3e} {o['clean']:10.1e} {o['Jnoise_aniso']:8.2f}")
with open(f'{OUT}/table4_baseline.csv', 'w') as fc:
    fc.write('variant,mean,std,worst_case,clean,Jnoise\n')
    for nm in ['A', 'D', 'B', 'C']:
        o = out[nm]; fc.write(f"{labs[nm]},{o['mean']:.6f},{o['std']:.6f},{o['wc']:.6f},{o['clean']:.2e},{o['Jnoise_aniso']:.4f}\n")

plt.figure(figsize=(8,5))                                          # Fig. 4
bins = np.linspace(0, max(infs['A'].max(), infs['C'].max())*1.02, 60)
plt.hist(infs['A'], bins=bins, alpha=.6, label='noise-blind ($\\lambda=0$)', color='tab:red')
plt.hist(infs['C'], bins=bins, alpha=.6, label='geometry-aware ($\\lambda=0.005$, $l_y=4$)', color='tab:blue')
plt.xlabel('gate infidelity'); plt.ylabel('count (1000 samples)'); plt.legend()
savefig('figure4_histogram')

plt.figure(figsize=(8,5))                                          # Fig. 6
bp = plt.boxplot([infs['A'], infs['D'], infs['B'], infs['C']], showfliers=False, patch_artist=True, whis=(5, 95))
plt.xticks([1, 2, 3, 4], BOX_LABELS)
for p, c in zip(bp['boxes'], ['tab:red', 'tab:purple', 'tab:green', 'tab:blue']): p.set_facecolor(c); p.set_alpha(.55)
plt.yscale('log'); plt.ylabel('gate infidelity (1000 validation samples)')
savefig('figure11_baseline_comparison')

for nm, fn, ttl in [('A', 'figure8a_controls_blind', 'noise-blind ($\\lambda=0$)'),          # Fig. 5
                    ('C', 'figure8b_controls_aware', 'geometry-aware ($\\lambda=0.005,\\ l_y=4$)')]:
    ux, uy, _ = ctrl[nm]
    plt.figure(figsize=(8,4.6))
    plt.step(tgrid[:-1], ux, where='post', label='$u_x(t)$', lw=1.8)
    plt.step(tgrid[:-1], uy, where='post', label='$u_y(t)$', lw=1.8)
    plt.xlabel('time $t$'); plt.ylabel('control amplitude'); plt.title(ttl); plt.legend()
    savefig(fn)

## Block 9 — Unitarity: CFM4 vs RK4 (Fig. 1)

In [ ]:
def run_unitarity():
    ux_C, uy_C, _ = ctrl['C']
    ts_all = np.sort(np.unique(np.concatenate([tgrid, tgrid[:-1]+h/2, ts_nodes])))
    rr = np.random.default_rng(2024)
    pz = sample_matern(rr, ts_all, SIG_Z, ELL, 1)[0]                      # drawn to keep the seed sequence
    px = np.clip(sample_matern(rr, ts_all, 1.0, ELL, 1)[0], -3, 3)*SIG_X
    py = np.clip(sample_matern(rr, ts_all, 1.0, ELL, 1)[0], -3, 3)*SIG_Y
    def Hfull(t):
        n = min(int(t/h), N-1)
        return 0.5*omega0*sz + ux_C[n]*(1+np.interp(t, ts_all, px))*sx + uy_C[n]*(1+np.interp(t, ts_all, py))*sy
    def e2x2(O):
        v = np.sqrt(abs(O[0,0])**2 + abs(O[0,1])**2)
        return np.cos(v)*I2 + (np.sin(v)/v)*O if v > 1e-14 else I2
    dc, drk = [], []; U = I2.copy()
    for n in range(N):
        H1 = Hfull(tgrid[n]+c1*h); H2 = Hfull(tgrid[n]+c2*h)
        U = e2x2(-1j*h*(a2*H1+a1*H2)) @ e2x2(-1j*h*(a1*H1+a2*H2)) @ U
        dc.append(np.linalg.norm(U.conj().T@U - I2))
    U = I2.copy()
    for n in range(N):
        t0 = tgrid[n]; F = lambda X, t: -1j*Hfull(t)@X
        k1 = F(U, t0); k2 = F(U+0.5*h*k1, t0+h/2); k3 = F(U+0.5*h*k2, t0+h/2); k4 = F(U+h*k3, t0+h)
        U = U + (h/6)*(k1+2*k2+2*k3+k4); drk.append(np.linalg.norm(U.conj().T@U - I2))
    r['unitarity'] = {'cfm4_max': float(max(dc)), 'rk4_final': float(drk[-1])}; save_res(r)
    plt.figure(figsize=(8,5))
    plt.semilogy(tgrid[1:], np.maximum(dc, 1e-17), label='CFM4 (structure-preserving)', lw=2)
    plt.semilogy(tgrid[1:], np.maximum(drk, 1e-17), label='RK4', lw=2)
    plt.xlabel('time $t$'); plt.ylabel(r'$\|U^\dagger U-I\|_F$'); plt.legend()
    savefig('figure1_structure_preservation')
    with open(f'{OUT}/table1_structure_preservation.csv', 'w') as fc:
        fc.write('cfm4_max_defect,rk4_final_defect\n'); fc.write(f'{max(dc):.3e},{drk[-1]:.3e}\n')
    print(r['unitarity'])
run_unitarity()

## Block 10 — Pareto frontier (Fig. 3) and robustness-bound test (Fig. 8, Corollary 5.1)

In [ ]:
lams = [0.0, 1e-4, 1e-3, 5e-3, 1e-2]; par = []; ctrls_p = {}
for l in lams:
    nm = 'A' if l == 0.0 else ('C' if l == 0.005 else f'P{l:g}')
    if nm not in VAR: VAR[nm] = (l, 1, 4, False)
    ux, uy, ci = get(nm); ctrls_p[l] = (ux, uy)
    par.append((l, float(Jnoise(ux, uy, 1, 4)), ci, float(evaluate(ux, uy, vn).mean())))
r['pareto'] = par; save_res(r)
plt.figure(figsize=(8,5))
plt.plot([p[1] for p in par], [p[2] for p in par], 'o-', lw=2, ms=8)
for p in par: plt.annotate(f'$\\lambda={p[0]:g}$', (p[1], p[2]), textcoords='offset points', xytext=(6,6), fontsize=10)
plt.xlabel(r'geometric noise cost $J_{\mathrm{noise}}$'); plt.ylabel('deterministic gate infidelity')
plt.yscale('symlog', linthresh=1e-6)
savefig('figure3_pareto_frontier')
with open(f'{OUT}/table2_pareto.csv', 'w') as fc:
    fc.write('lambda,Jnoise,clean_infidelity,mean_noisy_infidelity\n')
    for p in par: fc.write(f'{p[0]:g},{p[1]:.4f},{p[2]:.3e},{p[3]:.5f}\n')

plt.figure(figsize=(8,5)); maxr = 0.0
for l, col in zip(lams, ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple']):
    ux, uy = ctrls_p[l]; U0 = prop_cfm4(ux, uy, None, None, None)
    sub = tuple(None if p is None else (p[0][:300], p[1][:300]) for p in vn)
    Un = prop_cfm4(ux, uy, *sub)
    d = dF(Un[0][0], Un[1][0], Un[2][0], Un[3][0], U0[0][0,0], U0[1][0,0], U0[2][0,0], U0[3][0,0])
    jn = float(Jnoise(ux, uy, 1, 4))
    if jn > 0: maxr = max(maxr, float(np.max(d))/jn)
    plt.scatter([jn]*300, d, s=14, alpha=.3, color=col, label=f'$\\lambda={l:g}$')
xl = np.linspace(0, max(Jnoise(u[0], u[1], 1, 4) for u in ctrls_p.values())*1.1, 50)
plt.plot(xl, xl, 'k--', lw=2, label='$y=x$')
plt.xlabel(r'$J_{\mathrm{noise}}$'); plt.ylabel(r'$d_F(U(T,\omega),U_0(T))$'); plt.legend()
r['theorem'] = {'max_ratio_dF_over_Jnoise': maxr}; save_res(r)
savefig('figure5_theorem_verification')
print(f'max d_F / J_noise = {maxr:.4f}   (predicted bound: 0.3*sqrt(2) = {0.3*np.sqrt(2):.3f})')

## Block 11 — Noise-amplitude and correlation-length sweeps (Figs. 9, 10)

In [ ]:
ux_A, uy_A, _ = ctrl['A']; ux_C, uy_C, _ = ctrl['C']
rng_sw = np.random.default_rng(31415)
sig_list = [0.2, 0.6, 1.0, 1.6, 2.0]; ell_list = [0.1, 0.2, 0.5, 1.0]
sw_s = {'blind': [], 'aware': []}; sw_l = {'blind': [], 'aware': []}
for sg in sig_list:
    nn = make_noise(rng_sw, 300, sx_=sg)
    sw_s['blind'].append(float(evaluate(ux_A, uy_A, nn).mean())); sw_s['aware'].append(float(evaluate(ux_C, uy_C, nn).mean()))
for lc in ell_list:
    nn = make_noise(rng_sw, 300, sell=lc)
    sw_l['blind'].append(float(evaluate(ux_A, uy_A, nn).mean())); sw_l['aware'].append(float(evaluate(ux_C, uy_C, nn).mean()))
r['sweep_sigma'] = {'sigma': sig_list, **sw_s}; r['sweep_ell'] = {'ell': ell_list, **sw_l}; save_res(r)
for xs, sw, xlab, fn in [(sig_list, sw_s, r'noise amplitude $\sigma$', 'figure6_noise_amplitude'),
                         (ell_list, sw_l, r'correlation length $\ell$', 'figure7_correlation_length')]:
    plt.figure(figsize=(8,5))
    plt.semilogy(xs, sw['blind'], 'o-', color='tab:red', lw=2, label='noise-blind')
    plt.semilogy(xs, sw['aware'], 's-', color='tab:blue', lw=2, label='geometry-aware')
    plt.xlabel(xlab); plt.ylabel('mean gate infidelity'); plt.legend(); savefig(fn)
print('blind/aware ratio (amplitude sweep):', np.round(np.array(sw_s['blind'])/np.array(sw_s['aware']), 1))
print('blind/aware ratio (correlation sweep):', np.round(np.array(sw_l['blind'])/np.array(sw_l['aware']), 1))

## Block 12 — Anisotropy sweep (Figs. 12, 13)

In [ ]:
ly_list = [1, 2, 4, 8, 16]; ai = []; an = []
for lyv in ly_list:
    nm = 'B' if lyv == 1 else ('C' if lyv == 4 else f'L{lyv}')
    if nm not in VAR: VAR[nm] = (0.005, 1, lyv, False)
    ux, uy, ci = get(nm)
    ai.append(float(evaluate(ux, uy, vn).mean())); an.append(float(Jnoise(ux, uy, 1, lyv)))
r['anisotropy'] = {'ly': ly_list, 'mean_infid': ai, 'Jnoise': an}; save_res(r)
plt.figure(figsize=(7,4.6)); plt.plot(ly_list, ai, 'o-', lw=2, ms=8); plt.xscale('log', base=2)
plt.xlabel(r'anisotropy weight $l_y$'); plt.ylabel('mean gate infidelity'); savefig('figure10a_anisotropy_infidelity')
plt.figure(figsize=(7,4.6)); plt.plot(ly_list, an, 's-', color='tab:green', lw=2, ms=8); plt.xscale('log', base=2)
plt.xlabel(r'anisotropy weight $l_y$'); plt.ylabel(r'geometric noise cost $J_{\mathrm{noise}}$'); savefig('figure10b_anisotropy_noise')
print(dict(zip(ly_list, np.round(ai, 5))))

## Block 13 — Discrete adjoint gradient (Section 5.4)
Theorem 5.1 with the Fréchet derivative of each exponential evaluated by the block-triangular identity expm([[X, E], [0, X]]) = [[eˣ, Dexp_X[E]], [0, eˣ]] (Eqs. (45)–(46)). Remark 5.1: sampled objectives average per-realization adjoints, with noise-dependent control derivatives.

In [ ]:
def gate_cost_and_grad(ux, uy, hh, nz=None, ndx=None, ndy=None):
    Nn = len(ux); zf = lambda p, k: p[k] if p is not None else np.zeros(Nn)
    fx = [1.0 + zf(ndx, k) for k in (0, 1)]; fy = [1.0 + zf(ndy, k) for k in (0, 1)]
    Hs = [((0.5*omega0 + zf(nz, k))[:, None, None]*sz + (ux*fx[k])[:, None, None]*sx
           + (uy*fy[k])[:, None, None]*sy) for k in (0, 1)]
    Om1 = -1j*hh*(a1*Hs[0] + a2*Hs[1]); Om2 = -1j*hh*(a2*Hs[0] + a1*Hs[1])
    dO1x = (-1j*hh*(a1*fx[0] + a2*fx[1]))[:, None, None]*sx; dO1y = (-1j*hh*(a1*fy[0] + a2*fy[1]))[:, None, None]*sy
    dO2x = (-1j*hh*(a2*fx[0] + a1*fx[1]))[:, None, None]*sx; dO2y = (-1j*hh*(a2*fy[0] + a1*fy[1]))[:, None, None]*sy
    def ef(X, E):                                  # exp(X) and Dexp_X[E] (Van Loan block identity)
        M = expm(np.block([[X, E], [np.zeros_like(X), X]])); return M[:, :2, :2], M[:, :2, 2:]
    E1, D1x = ef(Om1, dO1x); _, D1y = ef(Om1, dO1y)
    E2, D2x = ef(Om2, dO2x); _, D2y = ef(Om2, dO2y)
    Q = E2 @ E1; dQx = D2x @ E1 + E2 @ D1x; dQy = D2y @ E1 + E2 @ D1y
    U = np.empty((Nn+1, 2, 2), complex); U[0] = I2
    for n in range(Nn): U[n+1] = Q[n] @ U[n]                       # forward sweep
    z = np.trace(V.conj().T @ U[Nn]); J = 1.0 - abs(z)/2.0
    P = -0.5*(z/abs(z))*V                                           # terminal condition (48), d = 2
    gx = np.empty(Nn); gy = np.empty(Nn)
    for n in range(Nn-1, -1, -1):                                   # backward (adjoint) sweep
        Ph = P.conj().T
        gx[n] = np.real(np.trace(Ph @ dQx[n] @ U[n])); gy[n] = np.real(np.trace(Ph @ dQy[n] @ U[n]))
        P = Q[n].conj().T @ P
    return J, gx, gy

def full_cost_and_grad(u, lam, lx, ly, hh, noise=None):
    Nn = len(u)//2; ux, uy = u[:Nn], u[Nn:]
    if noise is None: samples = [(None, None, None)]
    else:
        S = max(p[0].shape[0] for p in noise if p is not None)
        pick = lambda p, s: None if p is None else (p[0][s], p[1][s])
        samples = [tuple(pick(p, s) for p in noise) for s in range(S)]
    J = 0.0; gx = np.zeros(Nn); gy = np.zeros(Nn)
    for smp in samples:
        Js, a, b = gate_cost_and_grad(ux, uy, hh, *smp); J += Js; gx += a; gy += b
    k = len(samples); J /= k; gx /= k; gy /= k
    if lam > 0:
        D = np.sqrt(lx*ux**2 + ly*uy**2); Ds = np.where(D > 1e-14, D, 1.0); on = D > 1e-14
        J += lam*hh*np.sum(D); gx += lam*hh*lx*ux/Ds*on; gy += lam*hh*ly*uy/Ds*on
    return J, np.concatenate([gx, gy])

def fd_J_batch_factory(lam, lx, ly, noise=None):
    nz, ndx, ndy = noise if noise is not None else (None, None, None)
    def J_batch(ub):
        Ux, Uy = ub[:, :N], ub[:, N:]
        return infid(*prop_cfm4(Ux, Uy, nz, ndx, ndy)).mean(axis=1) + lam*Jnoise(Ux, Uy, lx, ly)
    return J_batch

def adam_adjoint(lam, lx, ly, noise=None, iters=450, lr0=0.05):
    u = np.concatenate([0.5*np.sin(2*np.pi*tgrid[:-1]/T), 0.5*np.cos(2*np.pi*tgrid[:-1]/T)])
    m = np.zeros(2*N); v = np.zeros(2*N); b1, b2 = 0.9, 0.999
    for it in range(1, iters+1):
        _, g = full_cost_and_grad(u, lam, lx, ly, h, noise)
        lr = lr0*(0.3 if it > iters*0.6 else 1.0)*(0.1 if it > iters*0.85 else 1.0)
        m = b1*m + (1-b1)*g; v = b2*v + (1-b2)*g*g
        u = np.clip(u - lr*(m/(1-b1**it))/(np.sqrt(v/(1-b2**it)) + 1e-12), -2.0, 2.0)
    return u
print('adjoint functions defined')

## Block 14 — Adjoint gradients: verification, cost and re-optimization (Sec. 6.5, Fig. 7)
Timings depend on your hardware; the linear vs quadratic scaling does not.

In [ ]:
adj = {'gradcheck': {}, 'timing': {'N': [], 'adjoint_s': [], 'fd_s': []}, 'reopt': {}}
rng_g = np.random.default_rng(4242)
for name, lam, lx, ly, noise in [('surrogate', 0.005, 1, 4, None), ('noise-blind', 0.0, 1, 1, None), ('ensemble', 0.0, 1, 1, tr_noise)]:
    u = np.clip(0.5*rng_g.standard_normal(2*N), -2, 2)
    _, g = full_cost_and_grad(u, lam, lx, ly, h, noise)
    Jb = fd_J_batch_factory(lam, lx, ly, noise); eps = 1e-6
    Jv = Jb(np.vstack([u[None,:] + eps*np.eye(2*N), u[None,:] - eps*np.eye(2*N)]))
    gfd = (Jv[:2*N] - Jv[2*N:])/(2*eps)
    adj['gradcheck'][name] = float(np.linalg.norm(g - gfd)/np.linalg.norm(gfd))
    if name == 'surrogate': g_plot, gfd_plot = g, gfd
    print(f'gradient check {name:12s}: relative error {adj["gradcheck"][name]:.2e}')

def time_gradients(Nt):
    global N, h
    saved = (N, h); N, h = Nt, T/Nt
    try:
        u = np.clip(0.5*np.random.default_rng(7).standard_normal(2*N), -2, 2)
        t0 = time.perf_counter(); full_cost_and_grad(u, 0.005, 1, 4, h, None); ta = time.perf_counter()-t0
        Jb = fd_J_batch_factory(0.005, 1, 4, None)
        t0 = time.perf_counter(); Jb(np.vstack([u[None,:] + 3e-7*np.eye(2*N), u[None,:]])); tf = time.perf_counter()-t0
    finally:
        N, h = saved
    return ta, tf
for Nt in ([50, 100, 200] if FAST else [50, 100, 200, 400, 800]):
    ta, tf = time_gradients(Nt)
    adj['timing']['N'].append(Nt); adj['timing']['adjoint_s'].append(ta); adj['timing']['fd_s'].append(tf)
    print(f'N = {Nt:4d}: adjoint {ta*1e3:8.1f} ms | finite differences {tf*1e3:8.1f} ms | speed-up {tf/ta:5.1f}')

for nm in ['A', 'B', 'C', 'D']:
    lam, lx, ly, ens = VAR[nm]
    t0 = time.perf_counter(); u = adam_adjoint(lam, lx, ly, tr_noise if ens else None, iters=ITERS_D if ens else ITERS)
    w = time.perf_counter()-t0; ux, uy = u[:N], u[N:]
    adj['reopt'][nm] = dict(**stats(evaluate(ux, uy, vn)), wall_s=w)
    dev = max(abs(adj['reopt'][nm][k] - out[nm][k])/abs(out[nm][k]) for k in ('mean', 'std', 'wc'))
    print(f'{nm}: adjoint re-optimization mean {adj["reopt"][nm]["mean"]:.4e} (finite differences {out[nm]["mean"]:.4e}); '
          f'max relative deviation {dev:.1e}; {w:.1f} s')
json.dump(adj, open(f'{OUT}/adjoint_results.json', 'w'), indent=1)

from matplotlib.ticker import NullFormatter, FixedLocator, FixedFormatter
fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))
ax[0].plot(gfd_plot, g_plot, '.', ms=4, alpha=.6, color='tab:blue', label='adjoint vs. finite-difference\ngradient components\n($2N=400$, anisotropic surrogate)'); lim = np.max(np.abs(gfd_plot))*1.05
ax[0].plot([-lim, lim], [-lim, lim], 'k--', lw=1.2, label='$y=x$')
ax[0].set_xlabel('central finite-difference gradient'); ax[0].set_ylabel('adjoint gradient'); ax[0].legend(loc='lower right', markerscale=3, fontsize=10); ax[0].set_title('(a) gradient verification')
tm = adj['timing']; Ns = np.array(tm['N'], float)
ax[1].loglog(Ns, tm['fd_s'], 'o-', lw=2, color='tab:red', label='batched finite differences')
ax[1].loglog(Ns, tm['adjoint_s'], 's-', lw=2, color='tab:blue', label='discrete adjoint')
ax[1].loglog(Ns, tm['adjoint_s'][0]*Ns/Ns[0], 'k:', lw=1, label=r'$\propto N$')
ax[1].loglog(Ns, tm['fd_s'][0]*(Ns/Ns[0])**2, 'k--', lw=1, label=r'$\propto N^2$')
ax[1].xaxis.set_major_locator(FixedLocator(tm['N'])); ax[1].xaxis.set_major_formatter(FixedFormatter([str(n) for n in tm['N']]))
ax[1].xaxis.set_minor_formatter(NullFormatter())
ax[1].set_xlabel('number of time steps $N$'); ax[1].set_ylabel('wall-clock time per gradient [s]'); ax[1].legend(); ax[1].set_title('(b) cost per gradient evaluation')
plt.tight_layout(); savefig('figure13_adjoint')

## Block 15 — Additive dephasing scenario (Sec. 6.10, Fig. 14)
Same four strategies, trained and validated with additive σz dephasing (σ = 0.05), which violates the control-coupled premise of the noise metric.

In [ ]:
set_scenario('Z'); rz = load_res(); vnz = val_noise(); outz = {}; infz = {}
for nm in ['A', 'B', 'C', 'D']:
    ux, uy, ci = get(nm); iv = evaluate(ux, uy, vnz); infz[nm] = iv
    outz[nm] = dict(**stats(iv), clean=ci, Jnoise_aniso=float(Jnoise(ux, uy, 1, 4)))
    print(f'{nm}: mean {outz[nm]["mean"]:.3e}  worst {outz[nm]["wc"]:.3e}')
rz['baseline'] = outz; save_res(rz)
plt.figure(figsize=(8,5))
bp = plt.boxplot([infz['A'], infz['D'], infz['B'], infz['C']], showfliers=False, patch_artist=True, whis=(5, 95))
plt.xticks([1, 2, 3, 4], BOX_LABELS)
for p, c in zip(bp['boxes'], ['tab:red', 'tab:purple', 'tab:green', 'tab:blue']): p.set_facecolor(c); p.set_alpha(.55)
plt.yscale('log'); plt.ylabel('gate infidelity (1000 validation samples)')
savefig('figure12_dephasing_baseline')
set_scenario('M'); r = load_res()                     # back to the main scenario

## Block 16 — Two-qubit controlled-Z gate under correlated dephasing (Sec. 6.8, Fig. 11)
JAX + L-BFGS-B, exact stepwise exponentials, Fourier controls (5 harmonics per line), objective = mean infidelity over 5 training realizations per qubit + λ J_noise (λ = 10⁻⁴, l_y = 4); up to ten initial guesses, stopping at the first with noiseless infidelity below 10⁻². Monte Carlo over 1000 validation realizations (seed disjoint from training). Takes about 1–3 minutes.

**Part A** evaluates the optimized Fourier coefficients reported in the manuscript; this is a plain forward simulation and reproduces Fig. 11 and the values in Sec. 6.8 exactly on any machine. **Part B** re-runs the optimization from scratch. L-BFGS trajectories on this non-convex landscape are sensitive to round-off (a one-ulp difference at an early line-search point is amplified over 400 iterations), so Part B may return a slightly different local optimum on different hardware or software versions; in our tests it gave a noiseless fidelity of 0.99971 and a mean noisy fidelity of 0.99729.

In [ ]:
def run_cz():
    import jax
    jax.config.update('jax_enable_x64', True)
    import jax.numpy as jnp
    from jax.scipy.linalg import expm as jexpm
    from scipy.optimize import minimize
    K = lambda A, B: jnp.array(np.kron(A, B))
    SX1, SY1, SX2, SY2 = K(sx, I2), K(sy, I2), K(I2, sx), K(I2, sy)
    SZ1, SZ2, HD = K(sz, I2), K(I2, sz), 0.5*K(sz, sz)
    CZ = jnp.diag(jnp.array([1, 1, 1, -1], dtype=jnp.complex128))
    T2, N2 = 10.0, 150; tg2 = np.linspace(0, T2, N2); h2 = T2/N2; NF = 5
    LAM, LX, LY, SIG, EL, CLIP = 1e-4, 1.0, 4.0, 0.05, 0.4, 3.0
    def noise(rng, S):
        L = np.linalg.cholesky(matern_cov(tg2, 1.0, EL))
        return SIG*np.clip((L @ rng.standard_normal((N2, S))).T, -CLIP, CLIP)
    _sin = jnp.array([np.sin((k+1)*np.pi*tg2/T2) for k in range(NF)])
    _cos = jnp.array([np.cos((k+1)*np.pi*tg2/T2) for k in range(NF)])
    def pulses(c):
        c = c.reshape(4, NF, 2); return c[:, :, 0] @ _sin + c[:, :, 1] @ _cos   # u1x, u1y, u2x, u2y
    def propagate(u, e1, e2):
        def step(U, k):
            H = HD + u[0,k]*SX1 + u[1,k]*SY1 + u[2,k]*SX2 + u[3,k]*SY2 + e1[k]*SZ1 + e2[k]*SZ2
            return jexpm(-1j*h2*H) @ U, None
        return jax.lax.scan(step, jnp.eye(4, dtype=jnp.complex128), jnp.arange(N2))[0]
    infid4 = lambda U: 1.0 - jnp.abs(jnp.trace(CZ.conj().T @ U))/4.0
    def objective(c, n1, n2):
        u = pulses(c)
        gate = jnp.mean(jax.vmap(lambda a, b: infid4(propagate(u, a, b)))(n1, n2))
        return gate + LAM*h2*jnp.sum(jnp.sqrt(LX*(u[0]**2 + u[2]**2) + LY*(u[1]**2 + u[3]**2)))
    obj_grad = jax.jit(jax.value_and_grad(objective))
    C_PUBLISHED = np.array([2.508823957103557, -3.0278491540424133, 0.8736451920339913, 2.924190990041513, -2.3884426159979335, 4.1821279554286415, -2.0195095748980565, 1.7926910567133112, 3.757289901597716, -3.451132915219039, 0.8760001681123077, -0.19853848953601969, -0.5197441279251268, 1.2412485417146424, -0.229236551474935, -1.4734220243412541, -2.3081253257293746, -1.9270410905347841, -0.925385065002649, -0.24410986625124728, 0.36735059921751306, -4.1478005949513275, -1.4226270448981093, 0.08837959867762823, 0.08753653657436017, 0.7970476835455667, -2.3988964507217494, -0.6363951437296809, 0.3318947963598948, -3.8826741709435644, -1.3017693121450196, 1.2277338566606428, 2.9817715551754715, -2.7750902616834385, 0.3284338921189994, 0.8557551311996633, 1.1884704320389212, 3.4029438608382123, -2.8891358520497645, -2.116375875791439])
    clean = jax.jit(lambda c: infid4(propagate(pulses(c), jnp.zeros(N2), jnp.zeros(N2))))
    batch = jax.jit(lambda c, n1, n2: jax.vmap(lambda a, b: infid4(propagate(pulses(c), a, b)))(n1, n2))
    rng_v = np.random.default_rng(777); S = 200 if FAST else 1000
    v1, v2 = jnp.array(noise(rng_v, S)), jnp.array(noise(rng_v, S))      # validation noise (disjoint seed)
    def evaluate_cz(c):
        F = 1.0 - np.array(batch(jnp.array(c), v1, v2)); ci = float(clean(jnp.array(c)))
        return F, dict(clean_fidelity=1-ci, clean_infidelity=ci, mc_samples=S, mc_mean=float(F.mean()),
                       mc_std=float(F.std()), mc_min=float(F.min()), mc_max=float(F.max()))
    # ---- Part A: published controls (exactly reproducible) ----
    F, cz = evaluate_cz(C_PUBLISHED)
    print('Part A, published controls :', {k: round(v, 6) if isinstance(v, float) else v for k, v in cz.items()})
    # ---- Part B: re-optimization from scratch (round-off sensitive) ----
    rng_tr = np.random.default_rng(4242)
    n1, n2 = jnp.array(noise(rng_tr, 5)), jnp.array(noise(rng_tr, 5))   # training noise
    wrap = lambda c: tuple(np.array(x, dtype=np.float64) for x in obj_grad(jnp.array(c), n1, n2))
    best = None
    for a in range(10):
        c0 = np.random.default_rng(999 + 7*a).uniform(-4.0, 4.0, 40)
        res = minimize(wrap, c0, method='L-BFGS-B', jac=True, bounds=[(-8.0, 8.0)]*40,
                       options={'maxiter': 40 if FAST else 400, 'ftol': 1e-9})
        ci = float(clean(jnp.array(res.x))); print(f'Part B, initial guess {a+1}: noiseless infidelity {ci:.3e}')
        if best is None or ci < best[1]: best = (res.x.copy(), ci)
        if best[1] < 1e-2: break
    _, czB = evaluate_cz(best[0])
    print('Part B, re-optimized here  :', {k: round(v, 6) if isinstance(v, float) else v for k, v in czB.items()})
    np.save(f'{OUT}/cz_coeffs_reoptimized.npy', best[0])
    json.dump({'published_controls': cz, 'reoptimized_here': czB}, open(f'{OUT}/cz_results.json', 'w'), indent=1)
    np.save(f'{OUT}/cz_mc_fidelities.npy', F)
    ci = cz['clean_infidelity']
    plt.figure(figsize=(8,5))                                          # Fig. 11 (published controls)
    plt.hist(F, bins=40, alpha=0.75, color='#8c564b', edgecolor='black')
    plt.axvline(F.mean(), color='black', ls='--', lw=2, label=f'mean $\\mathcal{{F}}$ = {F.mean():.5f}')
    plt.axvline(1-ci, color='tab:blue', ls=':', lw=2, label=f'noiseless $\\mathcal{{F}}$ = {1-ci:.5f}')
    plt.text(0.03, 0.95, f'{S} validation realizations\nmin $\\mathcal{{F}}$ = {F.min():.5f}\nstd = {F.std():.2e}',
             transform=plt.gca().transAxes, va='top', fontsize=11, bbox=dict(facecolor='white', alpha=0.85, edgecolor='black'))
    plt.xlabel(r'CZ gate fidelity $\mathcal{F}$'); plt.ylabel('Frequency')
    plt.title('Two-qubit CZ gate under correlated dephasing'); plt.legend(loc='center left', fontsize=10)
    savefig('figure9_CZ_histogram')
    cz['reoptimized'] = czB
    print(json.dumps(cz, indent=1)); return cz

cz = None
if RUN_CZ:
    try:
        import jax  # noqa: F401
        cz = run_cz()
    except ImportError:
        print('JAX is not installed: Block 16 skipped. Install with:  pip install "jax[cpu]"')

## Block 17 — Comparison with the values reported in the manuscript
Deterministic quantities should agree to the reported digits. Timings are hardware dependent and are not compared. Rows marked *info* (CZ re-optimization) are machine dependent at the 10⁻⁴ level, as explained in Block 16.

In [ ]:
rows = []
def chk(label, got, paper, tol):
    ok = abs(got - paper) <= tol*abs(paper)
    rows.append((label, got, paper, 'OK' if ok else 'CHECK'))
bl = r['baseline']
for nm, vals in {'A': (1.46e-2, 1.54e-2, 1.40e-1), 'D': (2.82e-3, 2.62e-3, 2.24e-2),
                 'B': (4.59e-3, 4.55e-3, 3.12e-2), 'C': (2.45e-3, 2.46e-3, 1.91e-2)}.items():
    for k, pv in zip(('mean', 'std', 'wc'), vals): chk(f'Table 1, {nm}, {k}', bl[nm][k], pv, 0.01)
    chk(f'Table 1, {nm}, J_noise', bl[nm]['Jnoise_aniso'], {'A': 4.48, 'D': 1.93, 'B': 1.93, 'C': 1.31}[nm], 0.01)
chk('Fig. 2, observed order (smooth)', fig2_slopes[0], 4.00, 0.02)
chk('Fig. 2, observed order (Matern)', fig2_slopes[1], 2.2, 0.05)
chk('Fig. 8, max d_F/J_noise', r['theorem']['max_ratio_dF_over_Jnoise'], 0.210, 0.01)
chk('Fig. 12, mean infidelity l_y=1', r['anisotropy']['mean_infid'][0], 4.6e-3, 0.02)
chk('Fig. 12, mean infidelity l_y=4', r['anisotropy']['mean_infid'][2], 2.45e-3, 0.01)
chk('Fig. 14, dephasing mean, D', outz['D']['mean'], 4.8e-3, 0.02)
chk('Fig. 14, dephasing worst, C', outz['C']['wc'], 4.4e-2, 0.02)
for nm, ge in adj['gradcheck'].items(): rows.append((f'Sec. 6.5, gradient check ({nm})', ge, 1e-8, 'OK' if ge < 1e-6 else 'CHECK'))
if cz is not None:
    chk('Fig. 11, CZ noiseless fidelity (published controls)', cz['clean_fidelity'], 0.99977, 1e-5)
    chk('Fig. 11, CZ mean noisy fidelity (published controls)', cz['mc_mean'], 0.99766, 1e-5)
    chk('Fig. 11, CZ worst case (published controls)', cz['mc_min'], 0.98910, 1e-5)
    rows.append(('Sec. 6.8, CZ re-optimized here: noiseless F', cz['reoptimized']['clean_fidelity'], 0.99977, 'info'))
    rows.append(('Sec. 6.8, CZ re-optimized here: mean F', cz['reoptimized']['mc_mean'], 0.99766, 'info'))
print(f"{'quantity':52s} {'reproduced':>12s} {'manuscript':>12s}  status")
for lab, g, p, s in rows: print(f'{lab:52s} {g:12.4e} {p:12.4e}  {s}')
print('\nAll figures and data are in:', OUT)
if FAST: print('FAST = True: deviations are expected. Set FAST = False for the paper values.')